#### Dynamic Prompts

In [ ]:
from langchain_community.utilities import SQLDatabase
import os
db = SQLDatabase.from_uri(os.getenv("DB_URL"))

In [ ]:
from dataclasses import dataclass
@dataclass
class RuntimeContext:
    is_employee: bool
    db: SQLDatabase

In [ ]:
from langchain_core.tools import tool
from langgraph.runtime import get_runtime

@tool
def execute_sql(sql: str) -> str:
    """Executes SQL Lite Query and return results"""
    runtime = get_runtime(RuntimeContext)
    db = runtime.context.db
    try:
        return db.run(sql)
    except Exception as e:
        return f"Error {e}"

In [ ]:
SYSTEM_PROMPT_TEMPLATE = """
You are a careful SQLite Analyst

Rules:
- Think Step By Step
- When you need data call the tool `execute_sql` with ONE SELECT query.
- Read-Only only: no INSERT/UPDATE/DELTE/TRUNCATE/DROP/CREATE/REPLACE/ALTER.
- Limit to 5 rows unless the user explicitly asks otherwise.
{table_limits}
- If the tool return 'Error'. Revise the SQL Try Again.
- prefer explicit column lists avoid SELECT *.
"""

In [ ]:
from langchain.agents.middleware.types import ModelRequest, dynamic_prompt

@dynamic_prompt
def dynamic_system_prompt(request: ModelRequest) -> str:
    if not request.runtime.context.is_employee:
        table_limits = "- Limit access to Album, Artist, Genere, Playlist, PlaylistTrack, Track"
    else:
        table_limits = ""
    return SYSTEM_PROMPT_TEMPLATE.format(table_limits = table_limits)

In [ ]:
from langchain.agents import create_agent
agent = create_agent(
    model="ollama:qwen2.5:7b",
    tools=[execute_sql],
    context_schema=RuntimeContext,
    middleware=[dynamic_system_prompt]
)

In [ ]:
question = "What is the most costly purchase by Frank Haris"

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    context=RuntimeContext(db=db,is_employee=True),
    stream_mode= "values"
    ):
    step["messages"][-1].pretty_print()